In [0]:
spark.conf.set(
    "fs.azure.account.key.storageppg.dfs.core.windows.net",
    "<account-key>"
)

from pyspark.sql import functions as F
from pyspark.sql.window import Window

base = "abfss://bronze@storageppg.dfs.core.windows.net/validated"

materials         = spark.read.option("header", True).csv(f"{base}/materials")
inventory         = spark.read.option("header", True).csv(f"{base}/inventory_snapshot")
purchase_orders   = spark.read.option("header", True).csv(f"{base}/purchase_orders")
production_orders = spark.read.option("header", True).csv(f"{base}/production_orders")
suppliers         = spark.read.option("header", True).csv(f"{base}/suppliers")
sales_orders      = spark.read.option("header", True).csv(f"{base}/sales_orders")

# Cast correct types
materials = materials.withColumn("unit_cost", F.col("unit_cost").cast("double")) \
                     .withColumn("reorder_level", F.col("reorder_level").cast("double")) \
                     .withColumn("lead_time_days", F.col("lead_time_days").cast("double"))

inventory = inventory.withColumn("quantity_on_hand", F.col("quantity_on_hand").cast("double")) \
                     .withColumn("snapshot_date", F.to_date("snapshot_date")) \
                     .withColumn("lot_expiry_date", F.to_date("lot_expiry_date"))

production_orders = production_orders.withColumn("quantity_consumed", F.col("quantity_consumed").cast("double")) \
                                     .withColumn("production_date", F.to_date("production_date"))

sales_orders = sales_orders.withColumn("quantity_ordered", F.col("quantity_ordered").cast("double")) \
                            .withColumn("order_date", F.to_date("order_date")) \
                            .withColumn("required_delivery_date", F.to_date("required_delivery_date"))

print("All files loaded and typed ✅")

All files loaded and typed ✅


# 12-month consumption per material

In [0]:
# Use latest snapshot date as reference point
latest_date = inventory.agg(F.max("snapshot_date")).collect()[0][0]
print(f"Reference date (latest snapshot): {latest_date}")

# 12-month window = last 365 days from latest production date
cutoff_date = F.add_months(F.lit(latest_date), -12)

consumption_12m = production_orders.filter(
    F.col("production_date") >= cutoff_date
).groupBy("material_id").agg(
    F.sum("quantity_consumed").alias("consumption_12m")
)

print("12-month consumption calculated ✅")
consumption_12m.show()

Reference date (latest snapshot): 2025-12-31
12-month consumption calculated ✅
+-----------+---------------+
|material_id|consumption_12m|
+-----------+---------------+
|     MAT004|         2280.0|
|     MAT017|         5640.0|
|     MAT012|         1080.0|
|     MAT003|         3280.0|
|     MAT018|         5040.0|
|     MAT002|         4860.0|
|     MAT011|         7705.0|
|     MAT001|        11800.0|
|     MAT006|        11980.0|
|     MAT016|         9840.0|
|     MAT013|         8200.0|
|     MAT015|         1080.0|
|     MAT008|         9700.0|
|     MAT020|         2800.0|
|     MAT007|         6050.0|
+-----------+---------------+



# Get latest inventory snapshot per material

In [0]:
# Get the most recent snapshot for each material
window_latest = Window.partitionBy("material_id").orderBy(F.desc("snapshot_date"))

latest_inventory = inventory.withColumn("rn", F.row_number().over(window_latest)) \
                             .filter(F.col("rn") == 1) \
                             .drop("rn")

print("Latest inventory snapshot per material ✅")
latest_inventory.show()

Latest inventory snapshot per material ✅
+-------------+-----------+------------------+----------------+---------------+
|snapshot_date|material_id|warehouse_location|quantity_on_hand|lot_expiry_date|
+-------------+-----------+------------------+----------------+---------------+
|   2025-12-31|     MAT001|              WH-A|           380.0|     2028-11-22|
|   2025-12-31|     MAT002|              WH-B|             0.0|     2028-02-26|
|   2025-12-31|     MAT003|              WH-C|             0.0|     2025-06-22|
|   2025-12-31|     MAT004|              WH-D|             0.0|     2028-05-19|
|   2025-12-31|     MAT005|              WH-E|           300.0|           NULL|
|   2025-12-31|     MAT006|              WH-A|             0.0|     2027-10-27|
|   2025-12-31|     MAT007|              WH-B|          3550.0|     2027-10-16|
|   2025-12-31|     MAT008|              WH-C|           240.0|     2027-09-03|
|   2025-12-31|     MAT009|              WH-D|           210.0|           NULL|

# RA flag (Recoverable Assets)

In [0]:
# Join inventory with 12-month consumption
inv_with_consumption = latest_inventory.join(consumption_12m, "material_id", "left") \
    .join(materials.select("material_id","material_name","category","unit_cost","reorder_level","lead_time_days"), 
          "material_id", "left") \
    .fillna(0, subset=["consumption_12m"])

# Flag: stock > 12-month consumption = Recoverable Asset
inv_with_consumption = inv_with_consumption.withColumn(
    "is_recoverable_asset",
    F.when(F.col("quantity_on_hand") > F.col("consumption_12m"), 1).otherwise(0)
)

print("Recoverable Asset flag applied ✅")
inv_with_consumption.select("material_id","material_name","quantity_on_hand","consumption_12m","is_recoverable_asset").show()

Recoverable Asset flag applied ✅
+-----------+-------------------+----------------+---------------+--------------------+
|material_id|      material_name|quantity_on_hand|consumption_12m|is_recoverable_asset|
+-----------+-------------------+----------------+---------------+--------------------+
|     MAT001|   Titanium Dioxide|           380.0|        11800.0|                   0|
|     MAT002|     Iron Oxide Red|             0.0|         4860.0|                   0|
|     MAT003|       Carbon Black|             0.0|         3280.0|                   0|
|     MAT004|Phthalocyanine Blue|             0.0|         2280.0|                   0|
|     MAT005|      Chrome Yellow|           300.0|            0.0|                   1|
|     MAT006|        Alkyd Resin|             0.0|        11980.0|                   0|
|     MAT007|      Acrylic Resin|          3550.0|         6050.0|                   0|
|     MAT008|        Epoxy Resin|           240.0|         9700.0|                   0|

# MC Dead Stock & MC Expired flags

In [0]:
# MC Dead Stock: no consumption for 9+ months AND no lot_expiry_date
cutoff_9m = F.add_months(F.lit(latest_date), -9)

consumption_9m = production_orders.filter(
    F.col("production_date") >= cutoff_9m
).groupBy("material_id").agg(
    F.sum("quantity_consumed").alias("consumption_9m")
)

inv_with_flags = inv_with_consumption.join(consumption_9m, "material_id", "left") \
    .fillna(0, subset=["consumption_9m"])

inv_with_flags = inv_with_flags.withColumn(
    "is_mc_dead_stock",
    F.when(
        (F.col("consumption_9m") == 0) & (F.col("lot_expiry_date").isNull()),
        1
    ).otherwise(0)
)

# MC Expired: lot_expiry_date has passed (before latest snapshot date)
inv_with_flags = inv_with_flags.withColumn(
    "is_mc_expired",
    F.when(
        F.col("lot_expiry_date") < F.lit(latest_date),
        1
    ).otherwise(0)
)

# Combined MC flag
inv_with_flags = inv_with_flags.withColumn(
    "is_magna_carta",
    F.when((F.col("is_mc_dead_stock") == 1) | (F.col("is_mc_expired") == 1), 1).otherwise(0)
)

print("MC Dead Stock & Expired flags applied ✅")
inv_with_flags.select(
    "material_id","material_name","lot_expiry_date",
    "consumption_9m","is_mc_dead_stock","is_mc_expired","is_magna_carta"
).show()

MC Dead Stock & Expired flags applied ✅
+-----------+-------------------+---------------+--------------+----------------+-------------+--------------+
|material_id|      material_name|lot_expiry_date|consumption_9m|is_mc_dead_stock|is_mc_expired|is_magna_carta|
+-----------+-------------------+---------------+--------------+----------------+-------------+--------------+
|     MAT001|   Titanium Dioxide|     2028-11-22|        9610.0|               0|            0|             0|
|     MAT002|     Iron Oxide Red|     2028-02-26|        3600.0|               0|            0|             0|
|     MAT003|       Carbon Black|     2025-06-22|        2380.0|               0|            1|             1|
|     MAT004|Phthalocyanine Blue|     2028-05-19|        1740.0|               0|            0|             0|
|     MAT005|      Chrome Yellow|           NULL|           0.0|               1|            0|             1|
|     MAT006|        Alkyd Resin|     2027-10-27|        9030.0|        

# MC Provision (100% financial write-down)

In [0]:
# Provision = quantity_on_hand × unit_cost for all MC-flagged materials
inv_with_flags = inv_with_flags.withColumn(
    "mc_provision_amount",
    F.when(
        F.col("is_magna_carta") == 1,
        F.col("quantity_on_hand") * F.col("unit_cost")
    ).otherwise(0.0)
)

total_provision = inv_with_flags.agg(
    F.sum("mc_provision_amount").alias("total_mc_provision")
).collect()[0][0]

print(f"Total MC Provision Required: RM {total_provision:,.2f} ✅")
inv_with_flags.select(
    "material_id","material_name","quantity_on_hand",
    "unit_cost","is_magna_carta","mc_provision_amount"
).filter(F.col("is_magna_carta") == 1).show()

Total MC Provision Required: RM 42,772.00 ✅
+-----------+------------------+----------------+---------+--------------+-------------------+
|material_id|     material_name|quantity_on_hand|unit_cost|is_magna_carta|mc_provision_amount|
+-----------+------------------+----------------+---------+--------------+-------------------+
|     MAT003|      Carbon Black|             0.0|      2.8|             1|                0.0|
|     MAT005|     Chrome Yellow|           300.0|      6.4|             1|             1920.0|
|     MAT009|Polyurethane Resin|           210.0|     11.2|             1|             2352.0|
|     MAT010|       Vinyl Resin|          2800.0|      7.6|             1|            21280.0|
|     MAT014|           Acetone|          5200.0|      2.1|             1|            10920.0|
|     MAT019|     Cardboard Box|         14000.0|     0.45|             1|             6300.0|
+-----------+------------------+----------------+---------+--------------+-------------------+



# Stockout risk flag (Q1 & Q6)

In [0]:
# Stockout risk: quantity_on_hand < (avg_daily_consumption × lead_time_days)
# avg_daily_consumption = consumption_12m / 365

inv_with_flags = inv_with_flags.withColumn(
    "avg_daily_consumption",
    F.col("consumption_12m") / 365
)

inv_with_flags = inv_with_flags.withColumn(
    "safety_stock_needed",
    F.col("avg_daily_consumption") * F.col("lead_time_days")
)

# Q1: below reorder level
inv_with_flags = inv_with_flags.withColumn(
    "is_below_reorder",
    F.when(F.col("quantity_on_hand") < F.col("reorder_level"), 1).otherwise(0)
)

# Stockout risk: stock won't cover lead time demand
inv_with_flags = inv_with_flags.withColumn(
    "is_stockout_risk",
    F.when(F.col("quantity_on_hand") < F.col("safety_stock_needed"), 1).otherwise(0)
)

print("Stockout risk flags applied ✅")
inv_with_flags.select(
    "material_id","material_name","quantity_on_hand",
    "reorder_level","safety_stock_needed",
    "is_below_reorder","is_stockout_risk"
).show()

Stockout risk flags applied ✅
+-----------+-------------------+----------------+-------------+-------------------+----------------+----------------+
|material_id|      material_name|quantity_on_hand|reorder_level|safety_stock_needed|is_below_reorder|is_stockout_risk|
+-----------+-------------------+----------------+-------------+-------------------+----------------+----------------+
|     MAT001|   Titanium Dioxide|           380.0|        500.0|  452.6027397260274|               1|               1|
|     MAT002|     Iron Oxide Red|             0.0|        300.0| 133.15068493150685|               1|               1|
|     MAT003|       Carbon Black|             0.0|        200.0|   62.9041095890411|               1|               1|
|     MAT004|Phthalocyanine Blue|             0.0|        150.0| 131.17808219178082|               1|               1|
|     MAT005|      Chrome Yellow|           300.0|        100.0|                0.0|               0|               0|
|     MAT006|     

# Sales order impact (Q7)

In [0]:
# Find open sales orders
open_orders = sales_orders.filter(F.col("status") == "Open")

# Get materials at stockout risk
stockout_materials = inv_with_flags.filter(F.col("is_stockout_risk") == 1) \
    .select("material_id", "material_name", "quantity_on_hand")

# Link: production_orders tells us which material goes into which finished_product
material_to_product = production_orders.select(
    "material_id", "finished_product"
).dropDuplicates()

# Join stockout materials → finished products → open sales orders
impacted_orders = stockout_materials \
    .join(material_to_product, "material_id", "inner") \
    .join(open_orders, "finished_product", "inner") \
    .select(
        "sales_order_id",
        "customer_name",
        "finished_product",
        "material_id",
        "material_name",
        "quantity_ordered",
        "required_delivery_date",
        "status",
        "quantity_on_hand"
    ).dropDuplicates()

print("Impacted open sales orders ✅")
impacted_orders.show(truncate=False)

Impacted open sales orders ✅
+--------------+------------------------+----------------------+-----------+-------------------+----------------+----------------------+------+----------------+
|sales_order_id|customer_name           |finished_product      |material_id|material_name      |quantity_ordered|required_delivery_date|status|quantity_on_hand|
+--------------+------------------------+----------------------+-----------+-------------------+----------------+----------------------+------+----------------+
|SO-040        |Sime Darby Property     |Premium White Paint 4L|MAT001     |Titanium Dioxide   |400.0           |2026-01-15            |Open  |380.0           |
|SO-048        |BuildRight Hardware     |Wood Stain Brown      |MAT002     |Iron Oxide Red     |225.0           |2026-01-05            |Open  |0.0             |
|SO-044        |Pavilion Retail Supplies|Industrial Grey Primer|MAT003     |Carbon Black       |364.0           |2026-01-11            |Open  |0.0             |
|SO-0

## Write all outputs to silver

In [0]:
silver_path = "abfss://silver@storageppg.dfs.core.windows.net/"

# Write as parquet (correct format)
inv_with_flags.write.format("parquet").mode("overwrite").save(silver_path + "inventory/")
impacted_orders.write.format("parquet").mode("overwrite").save(silver_path + "impacted_sales_orders/")
materials.write.format("parquet").mode("overwrite").save(silver_path + "materials/")
suppliers.write.format("parquet").mode("overwrite").save(silver_path + "suppliers/")
sales_orders.write.format("parquet").mode("overwrite").save(silver_path + "sales_orders/")
production_orders.write.format("parquet").mode("overwrite").save(silver_path + "production_orders/")

print("Silver layer written successfully ✅")
print("\nFiles written:")
print("  - silver/inventory/           ← inv_with_flags (all RA/MC flags)")
print("  - silver/impacted_sales_orders/ ← impacted orders")
print("  - silver/materials/           ← for dim_material")
print("  - silver/suppliers/           ← for dim_supplier")
print("  - silver/sales_orders/        ← for dim_sales_order")
print("  - silver/production_orders/   ← for dim_production")

Silver layer written successfully ✅

Files written:
  - silver/inventory/           ← inv_with_flags (all RA/MC flags)
  - silver/impacted_sales_orders/ ← impacted orders
  - silver/materials/           ← for dim_material
  - silver/suppliers/           ← for dim_supplier
  - silver/sales_orders/        ← for dim_sales_order
  - silver/production_orders/   ← for dim_production


## Summary report (verify your answers)

In [0]:
print("=" * 50)
print("PPG RA/MC SUMMARY REPORT")
print("=" * 50)

# Q1 - Which materials are below reorder level
q1 = inv_with_flags.filter(F.col('is_below_reorder') == 1) \
    .select('material_id', 'material_name', 'quantity_on_hand', 'reorder_level')
print(f"\nQ1 - Materials below reorder level ({q1.count()}):")
for row in q1.collect():
    print(f"     - {row['material_name']} | Stock: {row['quantity_on_hand']:,.2f} | Reorder Level: {row['reorder_level']:,.2f}")

# Q2 - Which materials are Recoverable Assets
q2 = inv_with_flags.filter(F.col('is_recoverable_asset') == 1) \
    .select('material_id', 'material_name', 'quantity_on_hand', 'consumption_12m')
print(f"\nQ2 - Recoverable Assets ({q2.count()}):")
for row in q2.collect():
    print(f"     - {row['material_name']} | Stock: {row['quantity_on_hand']:,.2f} | 12M Consumption: {row['consumption_12m']:,.2f}")

# Q3 - Which materials are MC Dead Stock
q3 = inv_with_flags.filter(F.col('is_mc_dead_stock') == 1) \
    .select('material_id', 'material_name', 'quantity_on_hand', 'consumption_9m')
print(f"\nQ3 - Magna Carta Dead Stock ({q3.count()}):")
for row in q3.collect():
    print(f"     - {row['material_name']} | Stock: {row['quantity_on_hand']:,.2f} | 9M Consumption: {row['consumption_9m']:,.2f}")

# Q4 - Which materials are MC Expired
q4 = inv_with_flags.filter(F.col('is_mc_expired') == 1) \
    .select('material_id', 'material_name', 'quantity_on_hand', 'lot_expiry_date')
print(f"\nQ4 - Magna Carta Expired ({q4.count()}):")
for row in q4.collect():
    print(f"     - {row['material_name']} | Stock: {row['quantity_on_hand']:,.2f} | Expiry Date: {row['lot_expiry_date']}")

# Q5 - Total MC Provision amount
q5_total = inv_with_flags.agg(F.sum('mc_provision_amount')).collect()[0][0]
q5_breakdown = inv_with_flags.filter(F.col('mc_provision_amount') > 0) \
    .select('material_name', 'quantity_on_hand', 'unit_cost', 'mc_provision_amount')
print(f"\nQ5 - Total Magna Carta Provision Required: RM {q5_total:,.2f}")
print(f"     Breakdown:")
for row in q5_breakdown.collect():
    print(f"     - {row['material_name']} | Stock: {row['quantity_on_hand']:,.2f} | Unit Cost: RM {row['unit_cost']:,.4f} | Provision: RM {row['mc_provision_amount']:,.2f}")

# Q6 - Materials at stockout risk affecting production
q6 = inv_with_flags.filter(F.col('is_stockout_risk') == 1) \
    .select('material_id', 'material_name', 'quantity_on_hand', 'avg_daily_consumption', 'safety_stock_needed')
print(f"\nQ6 - Materials at stockout risk affecting production ({q6.count()}):")
for row in q6.collect():
    print(f"     - {row['material_name']} | Stock: {row['quantity_on_hand']:,.2f} | Avg Daily Consumption: {row['avg_daily_consumption']:,.2f} | Safety Stock Needed: {row['safety_stock_needed']:,.2f}")

# Q7 - Which open sales orders and customers are impacted
q7_orders = impacted_orders.select(
    'sales_order_id', 'customer_name', 'material_name',
    'quantity_ordered', 'required_delivery_date', 'status', 'quantity_on_hand'
).orderBy('customer_name')
print(f"\nQ7 - Open sales orders impacted by stockout ({impacted_orders.select('sales_order_id').distinct().count()}):")
print(f"     Customers affected: {impacted_orders.select('customer_name').distinct().count()}")
for row in q7_orders.collect():
    print(f"     - Order: {row['sales_order_id']} | Customer: {row['customer_name']} | Material: {row['material_name']} | Qty Ordered: {row['quantity_ordered']:,.2f} | Stock Available: {row['quantity_on_hand']:,.2f} | Required: {row['required_delivery_date']} | Status: {row['status']}")

print("\n" + "=" * 50)
print("END OF REPORT")
print("=" * 50)

PPG RA/MC SUMMARY REPORT

Q1 - Materials below reorder level (9):
     - Titanium Dioxide | Stock: 380.00 | Reorder Level: 500.00
     - Iron Oxide Red | Stock: 0.00 | Reorder Level: 300.00
     - Carbon Black | Stock: 0.00 | Reorder Level: 200.00
     - Phthalocyanine Blue | Stock: 0.00 | Reorder Level: 150.00
     - Alkyd Resin | Stock: 0.00 | Reorder Level: 800.00
     - Epoxy Resin | Stock: 240.00 | Reorder Level: 400.00
     - Polyurethane Resin | Stock: 210.00 | Reorder Level: 300.00
     - Mineral Spirits | Stock: 0.00 | Reorder Level: 600.00
     - Plastic Drum 20L | Stock: 0.00 | Reorder Level: 500.00

Q2 - Recoverable Assets (10):
     - Chrome Yellow | Stock: 300.00 | 12M Consumption: 0.00
     - Polyurethane Resin | Stock: 210.00 | 12M Consumption: 0.00
     - Vinyl Resin | Stock: 2,800.00 | 12M Consumption: 0.00
     - Toluene | Stock: 7,880.00 | 12M Consumption: 1,080.00
     - Acetone | Stock: 5,200.00 | 12M Consumption: 0.00
     - Butanol | Stock: 2,580.00 | 12M Consum

# Load Gold Layer

In [0]:
from pyspark.sql import functions as F

silver_path = "abfss://silver@storageppg.dfs.core.windows.net/"
gold_path   = "abfss://gold@storageppg.dfs.core.windows.net/"

# Read Silver tables
silver_inventory  = spark.read.parquet(silver_path + "inventory/")
silver_materials  = spark.read.parquet(silver_path + "materials/")
silver_suppliers  = spark.read.parquet(silver_path + "suppliers/")
silver_sales      = spark.read.parquet(silver_path + "sales_orders/")
silver_production = spark.read.parquet(silver_path + "production_orders/")

# ---------- Dimension tables ----------
dim_material = silver_materials.select(
    "material_id", "material_name", "category",
    "unit_of_measure", "reorder_level", "lead_time_days", "unit_cost"
).dropDuplicates(["material_id"])

dim_supplier = silver_suppliers.select(
    "supplier_id", "supplier_name", "country", "reliability_rating"
).dropDuplicates(["supplier_id"])

dim_sales_order = silver_sales.select(
    "sales_order_id", "order_date", "customer_name",
    "finished_product", "quantity_ordered",
    "required_delivery_date", "status"
).dropDuplicates(["sales_order_id"])

dim_production = silver_production.select(
    "production_order_id", "production_date", "finished_product",
    "material_id", "quantity_consumed"
).dropDuplicates(["production_order_id"])

dim_date = silver_inventory.select("snapshot_date").dropDuplicates() \
    .withColumn("date_id",    F.date_format("snapshot_date", "yyyyMM").cast("int")) \
    .withColumn("year",       F.year("snapshot_date")) \
    .withColumn("quarter",    F.quarter("snapshot_date")) \
    .withColumn("month_name", F.date_format("snapshot_date", "MMMM"))

# ---------- Fact table ----------
fact = silver_inventory \
    .withColumn("date_id", F.date_format("snapshot_date", "yyyyMM").cast("int")) \
    .withColumn("fact_id",  F.monotonically_increasing_id().cast("int")) \
    .select(
        "fact_id", "material_id", "date_id",
        "quantity_on_hand", "consumption_12m", "consumption_9m",
        "reorder_level", "unit_cost", "mc_provision_amount",
        "avg_daily_consumption", "safety_stock_needed", "lot_expiry_date",
        "is_recoverable_asset", "is_mc_dead_stock", "is_mc_expired",
        "is_magna_carta", "is_below_reorder", "is_stockout_risk"
    )

# ---------- Write Gold tables ----------
dim_material.write.format("parquet").mode("overwrite").save(gold_path + "dim_material/")
dim_supplier.write.format("parquet").mode("overwrite").save(gold_path + "dim_supplier/")
dim_sales_order.write.format("parquet").mode("overwrite").save(gold_path + "dim_sales_order/")
dim_production.write.format("parquet").mode("overwrite").save(gold_path + "dim_production/")
dim_date.write.format("parquet").mode("overwrite").save(gold_path + "dim_date/")
fact.write.format("parquet").mode("overwrite").save(gold_path + "fact_inventory_status/")

print("Gold layer written successfully ✅")
print("\nTables written:")
print("  - gold/dim_material/")
print("  - gold/dim_supplier/")
print("  - gold/dim_sales_order/")
print("  - gold/dim_production/")
print("  - gold/dim_date/")
print("  - gold/fact_inventory_status/")

Gold layer written successfully ✅

Tables written:
  - gold/dim_material/
  - gold/dim_supplier/
  - gold/dim_sales_order/
  - gold/dim_production/
  - gold/dim_date/
  - gold/fact_inventory_status/
